# Prerequisite

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS merit_catalog;
USE CATALOG merit_catalog;
CREATE SCHEMA IF NOT EXISTS merit_catalog.quickstart_schema;


CREATE TABLE IF NOT EXISTS quickstart_schema.users_int(
   id INT,
   name STRING,
   dob DATE,
   email STRING,
   gender STRING,
   country STRING,
   region STRING,
   city STRING,
   asset INT,
   marital_status STRING
) USING DELTA;

DESCRIBE EXTENDED quickstart_schema.users_int;

# Transaction 01 - Loading the data into the table

In [0]:
df = spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
df.limit(4).display()

In [0]:
df.write.mode("overwrite").saveAsTable("quickstart_schema.users_int")

# Transaction 2

In [0]:
from pyspark.sql.functions import col

df.filter(col("country") == "India").write.mode("overwrite").saveAsTable(
    "quickstart_schema.users_int"
)

# Transaction 3

In [0]:
%sql
UPDATE quickstart_schema.users_int
SET country = "Bharat"
WHERE country = "India";

# Read the table

we can use `.option("versionAsOf", n)` method to specify the version

In [0]:
spark.read.table("quickstart_schema.users_int").display()

# Transaction Log

In [0]:
from delta.tables import DeltaTable
table_name = "quickstart_schema.users_int"
delta_table = DeltaTable.forName(spark,table_name)
history_df = delta_table.history()
history_df.display()

# Reading Specific version

## 1. Using PySpark API

In [0]:
spark.read.option("versionAsOf", 1).table("quickstart_schema.users_int").display()

## 2. Using SQL API

In [0]:
%sql
SELECT * from quickstart_schema.users_int VERSION AS OF 1;

# Read data based on Timestamp

## SQL

In [0]:
%sql
SELECT
  *
FROM
  quickstart_schema.users_int TIMESTAMP AS OF '2026-03-18T08:55';

# Restore 

In [0]:
%sql
RESTORE TABLE quickstart_schema.users_int TO VERSION AS OF 1;

In [0]:
history_df.display()